In [97]:
# import de todas las librerias que usaremos
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import seaborn as sns
pd.set_option('display.max_columns', None)
import math
from sklearn import linear_model
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from selenium import webdriver 
from selenium.webdriver.common.keys import Keys 
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
import time
from pandas import json_normalize
import requests
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [98]:
# Guardamos datos
pokemon = pd.read_csv('dataset\pokemon.csv')
pokemon_2= pd.read_csv('dataset\All_Pokemon.csv')

In [99]:
# Nos quedamos solo la 8 generacion
pokemon_2 = pokemon_2[pokemon_2["Generation"] == 8]

# Eliminamos los datos que no nos interesan y ponemos las columnas en el mismo formato para poder hacer concat
pokemon_2.drop(columns=["Mean","BMI","Mega Evolution","Alolan Form","Galarian Form","Standard Deviation","Experience type"],inplace= True)
pokemon_2.rename(columns={"Number":"pokedex_number","Att":"attack","Def":"defense","Spa":"sp_attack",
                          "Spd":"sp_defense","Spe":"speed","Experience to level 100":"experience_growth",
                          "Height":"height_m","Weight":"weight_kg","Catch Rate":"capture_rate","BST":"base_total",
                          "Type 1":"base_type","Type 2":"secondary_type","Against Fighting":"against_fight"}, inplace = True)

# Creamos columnas vacias con mismo nombre para concat
pokemon_2["base_egg_steps"] = 0
pokemon_2["base_happiness"] = 0

# Pasamos a minisculas las columnas
pokemon_2.columns = map(lambda x: str(x).lower(),pokemon_2.columns)

#Reemplazamos espacios en blancos
pokemon_2.columns=map(lambda x: str(x).replace(" ","_"),pokemon_2.columns)

# Reordenamos columnas
pokemon_2 = pokemon_2.loc[:,['pokedex_number','name','base_type', 'secondary_type','abilities', 'generation', 'legendary',
                         'attack', 'defense', 'hp', 'sp_attack', 'sp_defense', 'speed',
                         'against_bug', 'against_dark', 'against_dragon','against_electric',
                         'against_fairy', 'against_fight', 'against_fire','against_flying',
                         'against_ghost', 'against_grass', 'against_ground','against_ice', 
                         'against_normal', 'against_poison', 'against_psychic', 
                         'against_rock', 'against_steel', 'against_water', 
                         'base_egg_steps', 'base_happiness', 'base_total', 'capture_rate', 
                          'experience_growth','height_m','weight_kg'
                        ]]

# Capitalizamos las columnas
pokemon_2.columns = map(lambda x: str(x).capitalize(),pokemon_2.columns)

pokemon_2["Generation"] = pokemon_2.Generation.astype(int)
pokemon_2["Legendary"] = pokemon_2.Legendary.astype(int)

In [100]:
# Arreglamos para hacer concat con nuestro segundo dataset
pokemon.drop(columns=["japanese_name","percentage_male","classfication"], inplace=True)
# Reordenamos columnas
pokemon = pokemon.loc[:,['pokedex_number','name','type1', 'type2','abilities', 'generation', 'is_legendary',
                         'attack', 'defense', 'hp', 'sp_attack', 'sp_defense', 'speed',
                         'against_bug', 'against_dark', 'against_dragon','against_electric',
                         'against_fairy', 'against_fight', 'against_fire','against_flying',
                         'against_ghost', 'against_grass', 'against_ground','against_ice', 
                         'against_normal', 'against_poison', 'against_psychic', 
                         'against_rock', 'against_steel', 'against_water', 
                         'base_egg_steps', 'base_happiness', 'base_total', 'capture_rate', 
                          'experience_growth','height_m','weight_kg'
                        ]]

# Re nombramos algunas columnas para que sea mas claro
pokemon.rename(columns={"type1":"Base_type","type2":"Secondary_type","is_legendary":"Legendary"},inplace=True)

# Capitalizamos las columnas
pokemon.columns = map(lambda x: str(x).capitalize(),pokemon.columns)



In [101]:
# Hacemos concat de los dos Dataframes

pokemon = pd.concat([pokemon,pokemon_2])
pokemon.reset_index(drop=True)

,Pokedex_number,Name,Base_type,Secondary_type,Abilities,Generation,Legendary,Attack,Defense,Hp,Sp_attack,Sp_defense,Speed,Against_bug,Against_dark,Against_dragon,Against_electric,Against_fairy,Against_fight,Against_fire,Against_flying,Against_ghost,Against_grass,Against_ground,Against_ice,Against_normal,Against_poison,Against_psychic,Against_rock,Against_steel,Against_water,Base_egg_steps,Base_happiness,Base_total,Capture_rate,Experience_growth,Height_m,Weight_kg
0,1,Bulbasaur,grass,poison,"['Overgrow', 'Chlorophyll']",1,0,49,49,45,65,65,45,1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,0.25,1.0,2.0,1.0,1.0,2.0,1.0,1.0,0.5,5120,70,318,45,1059860,0.7,6.9
1,2,Ivysaur,grass,poison,"['Overgrow', 'Chlorophyll']",1,0,62,63,60,80,80,60,1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,0.25,1.0,2.0,1.0,1.0,2.0,1.0,1.0,0.5,5120,70,405,45,1059860,1.0,13.0
2,3,Venusaur,grass,poison,"['Overgrow', 'Chlorophyll']",1,0,100,123,80,122,120,80,1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,0.25,1.0,2.0,1.0,1.0,2.0,1.0,1.0,0.5,5120,70,625,45,1059860,2.0,100.0
3,4,Charmander,fire,NaN,"['Blaze', 'Solar Power']",1,0,52,43,39,60,50,65,0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,0.50,2.0,0.5,1.0,1.0,1.0,2.0,0.5,2.0,5120,70,309,45,1059860,0.6,8.5
4,5,Charmeleon,fire,NaN,"['Blaze', 'Solar Power']",1,0,64,58,58,80,65,80,0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,0.50,2.0,0.5,1.0,1.0,1.0,2.0,0.5,2.0,5120,70,405,45,1059860,1.1,19.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
911,896,Glastrier,Ice,NaN,['Chilling Neigh'],8,1,145,130,100,65,110,30,1.0,1.0,1.0,1.0,1.0,2.0,2.0,1.0,1.0,1.00,1.0,0.5,1.0,1.0,1.0,2.0,2.0,1.0,0,0,580,3,1250000,2.2,800.0
912,897,Spectrier,Ghost,NaN,['Grim Neigh'],8,1,65,60,100,145,80,130,0.5,2.0,1.0,1.0,1.0,0.0,1.0,1.0,2.0,1.00,1.0,1.0,0.0,0.5,1.0,1.0,1.0,1.0,0,0,580,3,1250000,2.0,44.5
913,898,Calyrex,Psychic,Grass,['Unnerve'],8,1,80,80,100,80,80,80,4.0,2.0,1.0,0.5,1.0,0.5,2.0,2.0,2.0,0.50,0.5,2.0,1.0,2.0,0.5,1.0,1.0,0.5,0,0,500,3,1250000,1.1,7.7
914,898,Calyrex Ice Rider,Psychic,Ice,['As One'],8,1,165,150,100,85,130,50,2.0,2.0,1.0,1.0,1.0,1.0,2.0,1.0,2.0,1.00,1.0,0.5,1.0,1.0,0.5,2.0,2.0,1.0,0,0,680,3,1250000,2.4,809.1


In [102]:
# Scrapeamos el numero de pokedex de todos los pokemons para poder hacer merge final de los datos y stats

PATH=("C:\Program Files (x86)\chromedriver.exe")
driver=webdriver.Chrome(PATH)
driver.get("https://pokemondb.net/pokedex/all")
time.sleep(2)
p = []
name = []
hp = []
attack = []
defense= []
spa =[]
spd =[]
speed = []
bt = []
# Looping to get pokedex number + stats
for i in range(1,1216):
    
    dex = driver.find_element(By.XPATH, f'//*[@id="pokedex"]/tbody/tr[{i}]/td[1]/span').text
    p.append(dex)
    print(dex)

    nombre = driver.find_element(By.XPATH, f'//*[@id="pokedex"]/tbody/tr[{i}]/td[2]/a').text
    name.append(nombre)
    print(nombre)
    

    total = driver.find_element(By.XPATH, f'//*[@id="pokedex"]/tbody/tr[{i}]/td[4]').text
    bt.append(total)
    print(total)
    

    vida = driver.find_element(By.XPATH, f'//*[@id="pokedex"]/tbody/tr[{i}]/td[5]').text
    hp.append(vida)
    print(vida)
    

    ataque = driver.find_element(By.XPATH, f'//*[@id="pokedex"]/tbody/tr[{i}]/td[6]').text
    attack.append(ataque)
    print(ataque)
    

    defensa = driver.find_element(By.XPATH, f'//*[@id="pokedex"]/tbody/tr[{i}]/td[7]').text
    defense.append(defensa)
    print(defensa)
    

    spat = driver.find_element(By.XPATH, f'//*[@id="pokedex"]/tbody/tr[{i}]/td[8]').text
    spa.append(spat)
    print(spat)
    
    
    sped = driver.find_element(By.XPATH, f'//*[@id="pokedex"]/tbody/tr[{i}]/td[9]').text
    spd.append(sped)
    print(sped)
       
  
    velocidad = driver.find_element(By.XPATH, f'//*[@id="pokedex"]/tbody/tr[{i}]/td[10]').text
    speed.append(velocidad)
    print(velocidad)
          
driver.close()


0001
Bulbasaur
318
45
49
49
65
65
45
0002
Ivysaur
405
60
62
63
80
80
60
0003
Venusaur
525
80
82
83
100
100
80
0003
Venusaur
625
80
100
123
122
120
80
0004
Charmander
309
39
52
43
60
50
65
0005
Charmeleon
405
58
64
58
80
65
80
0006
Charizard
534
78
84
78
109
85
100
0006
Charizard
634
78
130
111
130
85
100
0006
Charizard
634
78
104
78
159
115
100
0007
Squirtle
314
44
48
65
50
64
43
0008
Wartortle
405
59
63
80
65
80
58
0009
Blastoise
530
79
83
100
85
105
78
0009
Blastoise
630
79
103
120
135
115
78
0010
Caterpie
195
45
30
35
20
20
45
0011
Metapod
205
50
20
55
25
25
30
0012
Butterfree
395
60
45
50
90
80
70
0013
Weedle
195
40
35
30
20
20
50
0014
Kakuna
205
45
25
50
25
25
35
0015
Beedrill
395
65
90
40
45
80
75
0015
Beedrill
495
65
150
40
15
80
145
0016
Pidgey
251
40
45
40
35
35
56
0017
Pidgeotto
349
63
60
55
50
50
71
0018
Pidgeot
479
83
80
75
70
70
101
0018
Pidgeot
579
83
80
80
135
80
121
0019
Rattata
253
30
56
35
25
35
72
0019
Rattata
253
30
56
35
25
35
72
0020
Raticate
413
55
81
60
50
70
97

In [103]:
# Creamos un data Frame con lo scrapeado

new = {"Pokedex_number":p,"Name":name,"Base_total":bt,"Hp":hp,"Defense":defense,"Attack":attack,"Sp_attack":spa,"Sp_defense":spd,"Speed":speed}
scrap = pd.DataFrame(new)
scrap["Pokedex_number"] = scrap["Pokedex_number"].astype(int)
scrap["Base_total"] = scrap["Base_total"].astype(int)
scrap["Hp"] = scrap["Hp"].astype(int)
scrap["Defense"] = scrap["Defense"].astype(int)
scrap["Attack"] = scrap["Attack"].astype(int)
scrap["Sp_attack"] = scrap["Sp_attack"].astype(int)
scrap["Sp_defense"] = scrap["Sp_defense"].astype(int)
scrap["Speed"] = scrap["Speed"].astype(int)
scrap.drop_duplicates(subset='Name',keep='first',inplace=True)
scrap.tail()


,Pokedex_number,Name,Base_total,Hp,Defense,Attack,Sp_attack,Sp_defense,Speed
1208,1021,Raging Bolt,590,125,91,73,137,89,75
1209,1022,Iron Boulder,590,90,80,120,68,108,124
1210,1023,Iron Crown,590,90,100,72,122,108,98
1211,1024,Terapagos,450,90,85,65,65,85,60
1214,1025,Pecharunt,600,88,160,88,88,88,88


In [104]:
# Añadimos los pokemons que nos faltan al DF original
scrap = scrap.loc[1067:]
pokemon = pd.concat([pokemon,scrap])
pokemon.tail()

,Pokedex_number,Name,Base_type,Secondary_type,Abilities,Generation,Legendary,Attack,Defense,Hp,Sp_attack,Sp_defense,Speed,Against_bug,Against_dark,Against_dragon,Against_electric,Against_fairy,Against_fight,Against_fire,Against_flying,Against_ghost,Against_grass,Against_ground,Against_ice,Against_normal,Against_poison,Against_psychic,Against_rock,Against_steel,Against_water,Base_egg_steps,Base_happiness,Base_total,Capture_rate,Experience_growth,Height_m,Weight_kg
1208,1021,Raging Bolt,NaN,NaN,NaN,NaN,NaN,73,91,125,137,89,75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,590,NaN,NaN,NaN,NaN
1209,1022,Iron Boulder,NaN,NaN,NaN,NaN,NaN,120,80,90,68,108,124,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,590,NaN,NaN,NaN,NaN
1210,1023,Iron Crown,NaN,NaN,NaN,NaN,NaN,72,100,90,122,108,98,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,590,NaN,NaN,NaN,NaN
1211,1024,Terapagos,NaN,NaN,NaN,NaN,NaN,65,85,90,65,85,60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,450,NaN,NaN,NaN,NaN
1214,1025,Pecharunt,NaN,NaN,NaN,NaN,NaN,88,160,88,88,88,88,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,600,NaN,NaN,NaN,NaN


In [105]:
# Scrap de evoluciones de la API 

# Sacamos info adicional de la API de Pokemon
api_poke = []
poke_evol = []
conteo = 1
attempts = 0

# Usamos un while true para que hago mas intentos ya que hay casos en los que el status code era 400 y entonces paraba
while True:
    
    url = f"https://pokeapi.co/api/v2/evolution-chain/{conteo}"
    response = requests.get(url)
    if response.status_code != 200:
      attempts +=1
    else:
      # Guardamos el nombre del pokemon
        api_poke.append(response.json()["chain"]['species']["name"])

      # Guardaremos la evolucion de ese pokemon si tiene, si no, pondra que no tiene
        try:
          poke_evol.append(response.json()["chain"]["evolves_to"][0]["species"]["name"])
          print("Pokemon:", response.json()["chain"]['species']["name"])
        except:
            poke_evol.append("No hay")

    conteo +=1
          
    if attempts >= 15:
          
      break

print(len(api_poke),len(poke_evol))  

Pokemon: bulbasaur
Pokemon: charmander
Pokemon: squirtle
Pokemon: caterpie
Pokemon: weedle
Pokemon: pidgey
Pokemon: rattata
Pokemon: spearow
Pokemon: ekans
Pokemon: pichu
Pokemon: sandshrew
Pokemon: nidoran-f
Pokemon: nidoran-m
Pokemon: cleffa
Pokemon: vulpix
Pokemon: igglybuff
Pokemon: zubat
Pokemon: oddish
Pokemon: paras
Pokemon: venonat
Pokemon: diglett
Pokemon: meowth
Pokemon: psyduck
Pokemon: mankey
Pokemon: growlithe
Pokemon: poliwag
Pokemon: abra
Pokemon: machop
Pokemon: bellsprout
Pokemon: tentacool
Pokemon: geodude
Pokemon: ponyta
Pokemon: slowpoke
Pokemon: magnemite
Pokemon: farfetchd
Pokemon: doduo
Pokemon: seel
Pokemon: grimer
Pokemon: shellder
Pokemon: gastly
Pokemon: onix
Pokemon: drowzee
Pokemon: krabby
Pokemon: voltorb
Pokemon: exeggcute
Pokemon: cubone
Pokemon: tyrogue
Pokemon: lickitung
Pokemon: koffing
Pokemon: rhyhorn
Pokemon: happiny
Pokemon: tangela
Pokemon: horsea
Pokemon: goldeen
Pokemon: staryu
Pokemon: mime-jr
Pokemon: scyther
Pokemon: smoochum
Pokemon: elekid

In [106]:
# Pasamos las listas a DataFrame
evolutions = pd.DataFrame()
evolutions["Name"] = api_poke
evolutions["Evolution"] = poke_evol

# Ponemos los nombres en mayuscula para poder hacer el merge correctamente

evolutions["Name"] = evolutions["Name"].apply(lambda x: str(x).capitalize())
evolutions["Evolution"] = evolutions["Evolution"].apply(lambda x: str(x).capitalize())

# Hacemos merge de los dos Dataframes

pokemon = pokemon.merge(evolutions, left_on="Name", right_on="Name", how="left")

In [107]:
# Re indexamos las columnas para que sea mas facil trabajar

pokemon = pokemon.loc[:,['Pokedex_number', 'Name', 'Evolution', 'Base_type', 'Secondary_type',
       'Abilities', 'Generation', 'Legendary', 'Attack', 'Defense', 'Hp',
       'Sp_attack', 'Sp_defense', 'Speed', 'Height_m','Weight_kg', 'Against_bug', 'Against_dark',
       'Against_dragon', 'Against_electric', 'Against_fairy', 'Against_fight',
       'Against_fire', 'Against_flying', 'Against_ghost', 'Against_grass',
       'Against_ground', 'Against_ice', 'Against_normal', 'Against_poison',
       'Against_psychic', 'Against_rock', 'Against_steel', 'Against_water',
       'Base_egg_steps', 'Base_happiness', 'Base_total', 'Capture_rate',
       'Experience_growth'
        ]]

In [108]:
# Hacemos un fillna para poder filtrar por nulos y corregir las evoluciones

pokemon["Evolution"].fillna("Unknown",inplace=True)

# Corregimos evoluciones que faltaban

evol = {1:"Venusaur",
        4:"Charizard",
        7:"Blastoise",
        10:"Butterfree",
        13:"Beedrill",
        16:"Pidgeot",
        24:"Raichu",
        28:"Nidorina",
        29:"Nidoqueen",
        31:"Nidorino",
        32:"Nidoking",
        34:"Clefable",
        38:"Wigglytuff",
        41:"Crobat",
        43:"Vileplum",
        60:"Poliwrath",
        63:"Alakazam",
        66:"Machamp",
        69:"Victreebel",
        74:"Golem",
        92:"Gengar",
        116:"Kingdra",
        147:"Dragonite",
        153:"Meganium",
        155:"Typhlosion",
        158:"Feraligatr",
        182:"Azumarill",
        246:"Tyranitar",
        252:"Sceptile",
        256:"Blaziken",
        259:"Swampert",
        270:"Ludicolo",
        273:"Shiftry",
        280:"Gardevoir",
        287:"Slaking",
        290:"Shedinja",
        293:"Exploud",
        304:"Aggron",
        363:"Walrein",
        372:"Salamance",
        375:"Metagross",
        387:"Torterra",
        391:"Infernape",
        394:"Empoleon",
        396:"Staraptor",
        403:"Luxray",
        443:"Garchomp",
        495:"Serperior",
        498:"Emboar",
        501:"Samurott",
        507:"Stoutland",
        519:"Unfezant",
        525:"Gigalith",
        532:"Conkeldurr",
        535:"Seismitoad",
        540:"Leavanny",
        543:"Scolipede",
        551:"Krookodile",
        574:"Gothitelle",
        577:"Reuniclus",
        582:"Vanilluxe",
        599:"Klinklang",
        602:"Eelektross",
        607:"Chandelure",
        610:"Haxorus",
        650:"Chesnaught",
        653:"Delphox",
        656:"Greninja",
        661:"Talonflame",
        664:"Vivillion",
        668:"Floette",
        669:"Florges",
        679:"Aegislash",
        704:"Goodra",
        723:"Decidueye",
        726:"Incineroar",
        728:"Primarina",
        731:"Toucannon",
        736:"Vikavolt",
        761:"Tsareena",
        772:"Silvally",
        783:"Kommo-o",
        801:"Perrserker",
        802:"Galarian Rapidash",
        804: "Galarian Slowbro",
        806:"Galarian Sirfetch'd",
        807: "Galarian Koofing",
        808: "Galarian Mr.Rime",
        813: "Cursola",
        814: "Galarian Linoone",
        815: "Obstagoon",
        817: "Darmanitan",
        819: "Runerigus"


        }
pokemon["Evolution"].update(pd.Series(evol))

In [109]:
# Scrapeamos 9 generacion y sus tipos

PATH=("C:\Program Files (x86)\chromedriver.exe")
driver=webdriver.Chrome(PATH)
driver.get("https://bulbapedia.bulbagarden.net/wiki/List_of_Pok%C3%A9mon_by_National_Pok%C3%A9dex_number#List_of_Pok%C3%A9mon_by_National_Pok%C3%A9dex_number")
time.sleep(5)
new_gen = []
tipo1 = []
tipo2 = []

# Looping to get names
for i in range(2,136):
    try:

        name = driver.find_element_by_xpath(f'//*[@id="mw-content-text"]/div[1]/table[10]/tbody/tr[{i}]/td[3]/a').text
        new_gen.append(name)
        print(name)
        # Try to save first type
        try:
            t = driver.find_element_by_xpath(f'//*[@id="mw-content-text"]/div[1]/table[10]/tbody/tr[{i}]/td[4]').text
            tipo1.append(t)
            print(tipo1)
        
        except:
            tipo1.append(None)

    # Try to get second type
        try:
            
            t2 = driver.find_element_by_xpath(f'//*[@id="mw-content-text"]/div[1]/table[10]/tbody/tr[{i}]/td[5]').text
            tipo2.append(t2)
            print(tipo2)
        
        except:
            tipo2.append(None)
    except:
        driver.close()
driver.close()
# Convertimos a DataFrame la info
p = {"Name":new_gen,"Type_1":tipo1,"Type_2":tipo2}
scraped = pd.DataFrame(p)
scraped.tail()


Sprigatito
['Grass']
Floragato
['Grass', 'Grass']
Meowscarada
['Grass', 'Grass', 'Grass']
[None, None, 'Dark']
Fuecoco
['Grass', 'Grass', 'Grass', 'Fire']
Crocalor
['Grass', 'Grass', 'Grass', 'Fire', 'Fire']
Skeledirge
['Grass', 'Grass', 'Grass', 'Fire', 'Fire', 'Fire']
[None, None, 'Dark', None, None, 'Ghost']
Quaxly
['Grass', 'Grass', 'Grass', 'Fire', 'Fire', 'Fire', 'Water']
Quaxwell
['Grass', 'Grass', 'Grass', 'Fire', 'Fire', 'Fire', 'Water', 'Water']
Quaquaval
['Grass', 'Grass', 'Grass', 'Fire', 'Fire', 'Fire', 'Water', 'Water', 'Water']
[None, None, 'Dark', None, None, 'Ghost', None, None, 'Fighting']
Lechonk
['Grass', 'Grass', 'Grass', 'Fire', 'Fire', 'Fire', 'Water', 'Water', 'Water', 'Normal']
Oinkologne
['Grass', 'Grass', 'Grass', 'Fire', 'Fire', 'Fire', 'Water', 'Water', 'Water', 'Normal', 'Normal']
Normal
Tarountula
['Grass', 'Grass', 'Grass', 'Fire', 'Fire', 'Fire', 'Water', 'Water', 'Water', 'Normal', 'Normal', None, 'Bug']
Spidops
['Grass', 'Grass', 'Grass', 'Fire', 'Fir

,Name,Type_1,Type_2
129,Iron Boulder,Rock,Psychic
130,Iron Crown,Steel,Psychic
131,Terapagos,Normal,None
132,Normal,None,None
133,Pecharunt,Poison,Ghost


In [110]:
#Hacemos merge para poder juntar los types segun el nombre
pokemon = pokemon.merge(scraped, left_on="Name", right_on="Name", how="left")
gen9_type = scraped["Name"].to_list()
pokemon["Base_type"] = pokemon.apply(lambda x: x["Type_1"] if x["Name"] in gen9_type else x["Base_type"], axis=1)

pokemon["Secondary_type"] = pokemon.apply(lambda x: x["Type_2"] if x["Name"] in gen9_type else x["Secondary_type"], axis=1)



In [111]:
# Dropeamos las columnas que no necesitamos de type
pokemon.drop(columns=["Type_1","Type_2"], axis=1,inplace=True)

# Ponemos todos los tipos capitalizados
pokemon["Base_type"] = pokemon["Base_type"].apply(lambda x: str(x).capitalize())
pokemon["Secondary_type"] = pokemon["Secondary_type"].apply(lambda x: str(x).capitalize())

In [168]:
# Guardamos a csv

pokemon.to_csv('pokemon_casi.csv')
pokemon_casi = pd.read_csv('pokemon_casi.csv')

In [143]:
# Scrapeamos las habilidades y otros stats de la generacion 9

PATH=("C:\Program Files (x86)\chromedriver.exe")
driver=webdriver.Chrome(PATH)
driver.get("https://pokemondb.net/pokedex/all")
time.sleep(2)
numero = []
nombre = []
abilities = []
altura = []
peso = []
egg_step = []
capture = []
# Lo incializamos con este valor porque ahi empieza lo que nos interesa
contador = 899
# Generamos otro contador para ir haciendo append en la lista de listas para las habilidades
counter = 0
# Looping to get pokedex number + stats
for i in range(1070,1216):

    
    driver.find_element(By.XPATH, f'//*[@id="pokedex"]/tbody/tr[{i}]/td[2]/a').click()
    time.sleep(2)
    n = driver.find_element(By.XPATH, f'//*[@id="main"]/h1').text
    nombre.append(n)
    print(n)
    numero.append(i)
    abilities.append(i)
    print(i)

    # SI existe un pokemon con el mismo numero entonces scrapeamos primero el inicial y luego el siguiente:
    try:       
        # SI solo tiene una habilidad
        try:
            # Intentamos coger las habilidades segun el numero que tengan
            try:
                for z in range(1,3):    
                    ab = driver.find_element(By.XPATH,f'//*[@id="tab-basic-{contador}"]/div[1]/div[2]/table/tbody/tr[6]/td/span[{z}]/a').text
                    abilities.append(ab)
                    print(ab)
                   
                                
                ab2 = driver.find_element(By.XPATH,f'//*[@id="tab-basic-{contador}"]/div[1]/div[2]/table/tbody/tr[6]/td/small/a').text
                abilities.append(ab2)
                print(ab2)  

            except:

                ab = driver.find_element(By.XPATH,f'//*[@id="tab-basic-{contador}"]/div[1]/div[2]/table/tbody/tr[6]/td/span/a').text
                abilities.append(ab)
                print(ab)
                ab2 = driver.find_element(By.XPATH,f'//*[@id="tab-basic-{contador}"]/div[1]/div[2]/table/tbody/tr[6]/td/small/a').text
                abilities.append(ab2)
                print(ab2)
        except:

            ab = driver.find_element(By.XPATH, f'//*[@id="tab-basic-{contador}"]/div[1]/div[2]/table/tbody/tr[6]/td/span/a').text
            abilities.append(ab)
            print(ab)
            
            
        alt = driver.find_element(By.XPATH,f'//*[@id="tab-basic-{contador}"]/div[1]/div[2]/table/tbody/tr[4]/td').text
        altura.append(alt)
        print(alt)

        ps = driver.find_element(By.XPATH,f'//*[@id="tab-basic-{contador}"]/div[1]/div[2]/table/tbody/tr[5]/td').text
        peso.append(ps)
        print(ps)

        try:    
            eg = driver.find_element(By.XPATH,f'//*[@id="tab-basic-{contador}"]/div[1]/div[3]/div/div[2]/table/tbody/tr[3]/td/small').text
            egg_step.append(eg)
            print(eg)
        except:

            eg = driver.find_element(By.XPATH, f'//*[@id="tab-basic-{contador}"]/div[1]/div[3]/div/div[2]/table/tbody/tr[3]/td').text

        cap = driver.find_element(By.XPATH,f'//*[@id="tab-basic-{contador}"]/div[1]/div[3]/div/div[1]/table/tbody/tr[2]/td').text
        capture.append(cap)
        print(cap)
        
        contador+=1
    # SI existe un pokemon con el mismo numero entonces scrapeamos primero el inicial y luego el siguiente:
    except:
        contador-=1
        try:
        # Intentamos coger las habilidades segun el numero que tengan
            try:
                for z in range(1,3):    
                    ab = driver.find_element(By.XPATH,f'//*[@id="tab-basic-{contador}"]/div[1]/div[2]/table/tbody/tr[6]/td/span[{z}]/a').text
                    abilities.append(ab)
                    print(ab)
                    
                    
                ab2 = driver.find_element(By.XPATH,f'//*[@id="tab-basic-{contador}"]/div[1]/div[2]/table/tbody/tr[6]/td/small/a').text
                abilities.append(ab2)
                print(ab2)  

            except:

                ab = driver.find_element(By.XPATH,f'//*[@id="tab-basic-{contador}"]/div[1]/div[2]/table/tbody/tr[6]/td/span/a').text
                abilities.append(ab)
                print(ab)
                ab2 = driver.find_element(By.XPATH,f'//*[@id="tab-basic-{contador}"]/div[1]/div[2]/table/tbody/tr[6]/td/small/a').text
                abilities.append(ab2)
                print(ab2)
        except:

            ab = driver.find_element(By.XPATH, f'//*[@id="tab-basic-{contador}"]/div[1]/div[2]/table/tbody/tr[6]/td/span/a').text
            abilities.append(ab)
            print(ab)

        alt = driver.find_element(By.XPATH,f'//*[@id="tab-basic-{contador}"]/div[1]/div[2]/table/tbody/tr[4]/td').text
        altura.append(alt)
        print(alt)

        ps = driver.find_element(By.XPATH,f'//*[@id="tab-basic-{contador}"]/div[1]/div[2]/table/tbody/tr[5]/td').text
        peso.append(ps)
        print(ps)

        try:    
            eg = driver.find_element(By.XPATH,f'//*[@id="tab-basic-{contador}"]/div[1]/div[3]/div/div[2]/table/tbody/tr[3]/td/small').text
            egg_step.append(eg)
            print(eg)
        except:

            eg = driver.find_element(By.XPATH, f'//*[@id="tab-basic-{contador}"]/div[1]/div[3]/div/div[2]/table/tbody/tr[3]/td').text

        cap = driver.find_element(By.XPATH,f'//*[@id="tab-basic-{contador}"]/div[1]/div[3]/div/div[1]/table/tbody/tr[2]/td').text
        capture.append(cap)
        print(cap)

        
        contador+=1
   
        
    driver.back()
    time.sleep(1)

driver.close()


Calyrex
1070
Unnerve
Unnerve
Unnerve
1.1 m (3′07″)
7.7 kg (17.0 lbs)
(30,584–30,840 steps)
3 (0.4% with PokéBall, full HP)
Wyrdeer
1071
Intimidate
Frisk
Sap Sipper
1.8 m (5′11″)
95.1 kg (209.7 lbs)
(4,884–5,140 steps)
45 (5.9% with PokéBall, full HP)
Kleavor
1072
Swarm
Sheer Force
Sharpness
1.8 m (5′11″)
89.0 kg (196.2 lbs)
(4,884–5,140 steps)
15 (2.0% with PokéBall, full HP)
Ursaluna
1073
Guts
Bulletproof
Unnerve
2.4 m (7′10″)
290.0 kg (639.3 lbs)
(4,884–5,140 steps)
20 (2.6% with PokéBall, full HP)
Ursaluna
1074
Guts
Bulletproof
Unnerve
2.4 m (7′10″)
290.0 kg (639.3 lbs)
(4,884–5,140 steps)
20 (2.6% with PokéBall, full HP)
Basculegion
1075
Swift Swim
Adaptability
Mold Breaker
3.0 m (9′10″)
110.0 kg (242.5 lbs)
(4,884–5,140 steps)
45 (5.9% with PokéBall, full HP)
Basculegion
1076
Swift Swim
Adaptability
Mold Breaker
3.0 m (9′10″)
110.0 kg (242.5 lbs)
(4,884–5,140 steps)
45 (5.9% with PokéBall, full HP)
Sneasler
1077
Pressure
Unburden
Poison Touch
1.3 m (4′03″)
43.0 kg (94.8 lbs)
(4,88

In [144]:
# Convertimos la lista en lista de listas para mantener el formato del DF original
conteo = -1
listed_ab =[]
for i in abilities:
    if type(i) != int:
        listed_ab[conteo].append(i)
    else:
        listed_ab.append([])
        conteo +=1
        
# Check de lens
print(len(nombre))
print(len(listed_ab))
print(len(altura))
print(len(capture))
print(len(egg_step))
print(len(peso))

146
146
146
146
124
146


In [145]:
# Transformarmos los datos para mantener formato
altura = [i.split(' ')[0] for i in altura] 
peso = [i.split(' ')[0] for i in peso]
capture = [i.split(' ')[0] for i in capture]  
egg_step = [i[1:6] for i in egg_step]

In [146]:
# Replace de coma por punto
egg_step_2= [i.replace(',','.') for i in egg_step]

egg_step_2

['30.58',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '3.599',
 '3.599',
 '3.599',
 '3.599',
 '3.599',
 '4.884',
 '4.884',
 '3.599',
 '3.599',
 '3.599',
 '2.314',
 '2.314',
 '2.314',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '3.599',
 '3.599',
 '3.599',
 '3.599',
 '4.884',
 '4.884',
 '4.884',
 '8.739',
 '8.739',
 '8.739',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '8.739',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '8.739',
 '10.02',
 '10.02',
 '10.02',
 '4.884',
 '4.884',
 '7.454',
 '8.739',
 '7.454',
 '7.454',
 '4.884',
 '4.884',
 '4.884',
 '6.169',
 '6.169',
 '4.884',
 '10.02',
 '8.739',
 '8.739',
 '8.739',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '4.884',
 '12.59',
 '12.59',


In [147]:
# Llenamos para tener el mismo len
for i in range(22):
    egg_step_2.append(None)
len(egg_step_2)

146

In [169]:
# COMO CONVIERTO LO SCRAPEADO EN LISTA DE LISTAS PARA QUE TENGAN MISMO LENGHT?
sc = {"Pokedex":numero,"Name":nombre,"Ability":listed_ab,"Height":altura,"Weight":peso,"Base_egg_step":egg_step_2,"Capture":capture}
df = pd.DataFrame(sc)
df.head()

# Guardamos a csv
df.to_csv('9_gen_df.csv')
gen_df = pd.read_csv('9_gen_df.csv')

In [170]:
# ELiminamos las columnas no deseadas
gen_df.drop(columns='Unnamed: 0',inplace=True)
pokemon_casi.drop(columns='Unnamed: 0',inplace=True)
# Cambiamos types
gen_df["Weight"] = gen_df["Weight"].astype(float)
gen_df["Base_egg_step"] = gen_df["Base_egg_step"].astype(float)
gen_df["Height"] = gen_df["Height"].astype(int)

In [171]:
pokemon_casi.head()

,Pokedex_number,Name,Evolution,Base_type,Secondary_type,Abilities,Generation,Legendary,Attack,Defense,Hp,Sp_attack,Sp_defense,Speed,Height_m,Weight_kg,Against_bug,Against_dark,Against_dragon,Against_electric,Against_fairy,Against_fight,Against_fire,Against_flying,Against_ghost,Against_grass,Against_ground,Against_ice,Against_normal,Against_poison,Against_psychic,Against_rock,Against_steel,Against_water,Base_egg_steps,Base_happiness,Base_total,Capture_rate,Experience_growth
0,1,Bulbasaur,Ivysaur,Grass,Poison,"['Overgrow', 'Chlorophyll']",1.0,0.0,49,49,45,65,65,45,0.7,6.9,1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,0.25,1.0,2.0,1.0,1.0,2.0,1.0,1.0,0.5,5120.0,70.0,318,45,1059860.0
1,2,Ivysaur,Venusaur,Grass,Poison,"['Overgrow', 'Chlorophyll']",1.0,0.0,62,63,60,80,80,60,1.0,13.0,1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,0.25,1.0,2.0,1.0,1.0,2.0,1.0,1.0,0.5,5120.0,70.0,405,45,1059860.0
2,3,Venusaur,Unknown,Grass,Poison,"['Overgrow', 'Chlorophyll']",1.0,0.0,100,123,80,122,120,80,2.0,100.0,1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,0.25,1.0,2.0,1.0,1.0,2.0,1.0,1.0,0.5,5120.0,70.0,625,45,1059860.0
3,4,Charmander,Charmeleon,Fire,Nan,"['Blaze', 'Solar Power']",1.0,0.0,52,43,39,60,50,65,0.6,8.5,0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,0.50,2.0,0.5,1.0,1.0,1.0,2.0,0.5,2.0,5120.0,70.0,309,45,1059860.0
4,5,Charmeleon,Charizard,Fire,Nan,"['Blaze', 'Solar Power']",1.0,0.0,64,58,58,80,65,80,1.1,19.0,0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,0.50,2.0,0.5,1.0,1.0,1.0,2.0,0.5,2.0,5120.0,70.0,405,45,1059860.0


In [172]:
# Hacemos un merge por nombre para poder añadir el resto de features
pokemon_casi = pokemon_casi.merge(gen_df, left_on='Name', right_on='Name',how='left')

#Añadimos las features de los pokemons que hemos traido con el merge a nuestro df
gen9_stats = gen_df["Name"].to_list()
pokemon_casi["Abilities"] = pokemon_casi.apply(lambda x: x["Ability"] if x["Name"] in gen9_stats else x["Abilities"], axis=1)
pokemon_casi["Height_m"] = pokemon_casi.apply(lambda x: x["Height"] if x["Name"] in gen9_stats else x["Height_m"], axis=1)
pokemon_casi["Weight_kg"] = pokemon_casi.apply(lambda x: x["Weight"] if x["Name"] in gen9_stats else x["Weight_kg"], axis=1)
pokemon_casi["Base_egg_steps"] = pokemon_casi.apply(lambda x: x["Base_egg_step"] if x["Name"] in gen9_stats else x["Base_egg_steps"], axis=1)
pokemon_casi["Capture_rate"] = pokemon_casi.apply(lambda x: x["Capture"] if x["Name"] in gen9_stats else x["Capture_rate"], axis=1)

pokemon_casi

,Pokedex_number,Name,Evolution,Base_type,Secondary_type,Abilities,Generation,Legendary,Attack,Defense,Hp,Sp_attack,Sp_defense,Speed,Height_m,Weight_kg,Against_bug,Against_dark,Against_dragon,Against_electric,Against_fairy,Against_fight,Against_fire,Against_flying,Against_ghost,Against_grass,Against_ground,Against_ice,Against_normal,Against_poison,Against_psychic,Against_rock,Against_steel,Against_water,Base_egg_steps,Base_happiness,Base_total,Capture_rate,Experience_growth,Pokedex,Ability,Height,Weight,Base_egg_step,Capture
0,1,Bulbasaur,Ivysaur,Grass,Poison,"['Overgrow', 'Chlorophyll']",1.0,0.0,49,49,45,65,65,45,0.7,6.9,1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,0.25,1.0,2.0,1.0,1.0,2.0,1.0,1.0,0.5,5120.0,70.0,318,45,1059860.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Ivysaur,Venusaur,Grass,Poison,"['Overgrow', 'Chlorophyll']",1.0,0.0,62,63,60,80,80,60,1.0,13.0,1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,0.25,1.0,2.0,1.0,1.0,2.0,1.0,1.0,0.5,5120.0,70.0,405,45,1059860.0,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Venusaur,Unknown,Grass,Poison,"['Overgrow', 'Chlorophyll']",1.0,0.0,100,123,80,122,120,80,2.0,100.0,1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,0.25,1.0,2.0,1.0,1.0,2.0,1.0,1.0,0.5,5120.0,70.0,625,45,1059860.0,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Charmander,Charmeleon,Fire,Nan,"['Blaze', 'Solar Power']",1.0,0.0,52,43,39,60,50,65,0.6,8.5,0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,0.50,2.0,0.5,1.0,1.0,1.0,2.0,0.5,2.0,5120.0,70.0,309,45,1059860.0,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Charmeleon,Charizard,Fire,Nan,"['Blaze', 'Solar Power']",1.0,0.0,64,58,58,80,65,80,1.1,19.0,0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,0.50,2.0,0.5,1.0,1.0,1.0,2.0,0.5,2.0,5120.0,70.0,405,45,1059860.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1057,1023,Iron Crown,Unknown,Steel,Psychic,"['Quark Drive', 'Quark Drive', 'Quark Drive']",NaN,NaN,72,100,90,122,108,98,1.0,156.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,590,10.0,NaN,1211.0,"['Quark Drive', 'Quark Drive', 'Quark Drive']",1.0,156.0,NaN,10.0
1058,1024,Terapagos,No hay,Normal,NaN,"['Tera Shift', 'Tera Shift', 'Tera Shift']",NaN,NaN,65,85,90,65,85,60,0.0,6.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,450,255.0,NaN,1212.0,"['Tera Shift', 'Tera Shift', 'Tera Shift']",0.0,6.5,NaN,255.0
1059,1024,Terapagos,No hay,Normal,NaN,"['Tera Shift', 'Tera Shift', 'Tera Shift']",NaN,NaN,65,85,90,65,85,60,0.0,6.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,450,255.0,NaN,1213.0,"['Tera Shift', 'Tera Shift', 'Tera Shift']",0.0,6.5,NaN,255.0
1060,1024,Terapagos,No hay,Normal,NaN,"['Tera Shift', 'Tera Shift', 'Tera Shift']",NaN,NaN,65,85,90,65,85,60,0.0,6.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,450,255.0,NaN,1214.0,"['Tera Shift', 'Tera Shift', 'Tera Shift']",0.0,6.5,NaN,255.0


In [ ]:
# Dropeamos las columnas extra que nos ha traido el merge

pokemon_casi.drop(columns=["Pokedex","Ability","Height","Weight","Base_egg_step","Capture"],inplace=True)
pokemon_casi.tail()

In [177]:
# Fillna de secondary type
pokemon_casi["Secondary_type"].fillna('No more types',inplace= True)
# Replace de los Nan por el valor que toca
pokemon_casi["Secondary_type"] = pokemon_casi["Secondary_type"].apply(lambda x: 'No more types' if 'Nan' in x else x)

In [180]:
# Rellenamos campos que van faltando
pokemon_casi["Generation"].fillna(9.0,inplace=True)

In [ ]:
# Mostramos los duplicados para poder corregir
pokemon_casi[pokemon_casi.duplicated(keep=False)]

,Pokedex_number,Name,Evolution,Base_type,Secondary_type,Abilities,Generation,Legendary,Attack,Defense,Hp,Sp_attack,Sp_defense,Speed,Height_m,Weight_kg,Against_bug,Against_dark,Against_dragon,Against_electric,Against_fairy,Against_fight,Against_fire,Against_flying,Against_ghost,Against_grass,Against_ground,Against_ice,Against_normal,Against_poison,Against_psychic,Against_rock,Against_steel,Against_water,Base_egg_steps,Base_happiness,Base_total,Capture_rate,Experience_growth
919,901,Ursaluna,Unknown,Nan,No more types,"['Guts', 'Bulletproof', 'Unnerve']",9.0,NaN,140,105,130,45,80,50,2.0,290.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.884,NaN,550,20.0,NaN
920,901,Ursaluna,Unknown,Nan,No more types,"['Guts', 'Bulletproof', 'Unnerve']",9.0,NaN,140,105,130,45,80,50,2.0,290.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.884,NaN,550,20.0,NaN
921,902,Basculegion,Unknown,Nan,No more types,"['Swift Swim', 'Adaptability', 'Mold Breaker']",9.0,NaN,112,65,120,80,75,78,3.0,110.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.884,NaN,530,45.0,NaN
922,902,Basculegion,Unknown,Nan,No more types,"['Swift Swim', 'Adaptability', 'Mold Breaker']",9.0,NaN,112,65,120,80,75,78,3.0,110.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.884,NaN,530,45.0,NaN
925,905,Enamorus,No hay,Nan,No more types,"['Cute Charm', 'Cute Charm', 'Contrary']",9.0,NaN,115,70,74,135,80,106,1.0,48.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.884,NaN,580,3.0,NaN
926,905,Enamorus,No hay,Nan,No more types,"['Cute Charm', 'Cute Charm', 'Contrary']",9.0,NaN,115,70,74,135,80,106,1.0,48.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.884,NaN,580,3.0,NaN
937,916,Oinkologne,Unknown,Normal,No more types,"['Lingering Aroma', 'Gluttony', 'Thick Fat']",9.0,NaN,100,75,110,59,80,65,1.0,120.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.599,NaN,489,100.0,NaN
938,916,Oinkologne,Unknown,Normal,No more types,"['Lingering Aroma', 'Gluttony', 'Thick Fat']",9.0,NaN,100,75,110,59,80,65,1.0,120.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.599,NaN,489,100.0,NaN
947,925,Maushold,Unknown,Normal,No more types,"['Friend Guard', 'Cheek Pouch', 'Technician']",9.0,NaN,75,70,74,65,75,111,0.0,2.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.314,NaN,470,75.0,NaN
948,925,Maushold,Unknown,Normal,No more types,"['Friend Guard', 'Cheek Pouch', 'Technician']",9.0,NaN,75,70,74,65,75,111,0.0,2.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.314,NaN,470,75.0,NaN


In [ ]:
# Corregimos legendarios que faltaban

legend = {488:1,
          792:0,
          793:0,
          794:0,
          795:0,
          796:0,
          797:0,
          798:0,
          771:1,
          772:1,
          919:1,
          925:1,
          926:1,
          1032:1,
          1033:1,
          1034:1,
          1035:1,
          1038:1,
          1039:1,
          1045:1,
          1046:1,
          1047:1,
          1048:1,
          1049:1,
          1050:1,
          1051:1,
          1058:1,
          1059:1,
          1060:1}
pokemon_casi["Legendary"].update(pd.Series(legend))

# Adaptamos para no tener unknows o nulls en las Evoluciones

pokemon_casi["Evolution"].replace({"Unknown":"Final Form",
                              "No hay":"Final Form"}, inplace=True)

# Rellenamos con la media en Wheight i Height los nulls
pokemon_casi["Height_m"] = pokemon_casi["Height_m"].fillna(pokemon_casi["Height_m"].mean())
pokemon_casi["Weight_kg"] = pokemon_casi["Weight_kg"].fillna(pokemon_casi["Weight_kg"].mean())

# Corregimos algunos nombres

names = {920:'Ursalana Bloodmoon',
         921:'Basculegion Male',
         922:'Basculegion Female',
         925:,
         926:,
         937:,
         938:,
         947:,
         948:,
         954:,
         955:,
         956:,
         957:,
         990:,
         991:,
         1005:,
         1006:,
         1007:,
         1011:,
         1012:,
         1029:,
         1030:,
         1048:,
         1049:,
         1050:,
         1051:,
         1058:,
         1059:,
         1060:
         }

pokemon_casi["Name"].update(pd.Series(names))

,Pokedex_number,Name,Evolution,Base_type,Secondary_type,Abilities,Generation,Legendary,Attack,Defense,Hp,Sp_attack,Sp_defense,Speed,Height_m,Weight_kg,Against_bug,Against_dark,Against_dragon,Against_electric,Against_fairy,Against_fight,Against_fire,Against_flying,Against_ghost,Against_grass,Against_ground,Against_ice,Against_normal,Against_poison,Against_psychic,Against_rock,Against_steel,Against_water,Base_egg_steps,Base_happiness,Base_total,Capture_rate,Experience_growth
0,1,Bulbasaur,Ivysaur,Grass,Poison,"['Overgrow', 'Chlorophyll']",1.0,0.0,49,49,45,65,65,45,0.7,6.9,1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,0.25,1.0,2.0,1.0,1.0,2.0,1.0,1.0,0.5,5120.0,70.0,318,45,1059860.0
1,2,Ivysaur,Venusaur,Grass,Poison,"['Overgrow', 'Chlorophyll']",1.0,0.0,62,63,60,80,80,60,1.0,13.0,1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,0.25,1.0,2.0,1.0,1.0,2.0,1.0,1.0,0.5,5120.0,70.0,405,45,1059860.0
2,3,Venusaur,Unknown,Grass,Poison,"['Overgrow', 'Chlorophyll']",1.0,0.0,100,123,80,122,120,80,2.0,100.0,1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,0.25,1.0,2.0,1.0,1.0,2.0,1.0,1.0,0.5,5120.0,70.0,625,45,1059860.0
3,4,Charmander,Charmeleon,Fire,No more types,"['Blaze', 'Solar Power']",1.0,0.0,52,43,39,60,50,65,0.6,8.5,0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,0.50,2.0,0.5,1.0,1.0,1.0,2.0,0.5,2.0,5120.0,70.0,309,45,1059860.0
4,5,Charmeleon,Charizard,Fire,No more types,"['Blaze', 'Solar Power']",1.0,0.0,64,58,58,80,65,80,1.1,19.0,0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,0.50,2.0,0.5,1.0,1.0,1.0,2.0,0.5,2.0,5120.0,70.0,405,45,1059860.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1057,1023,Iron Crown,Unknown,Steel,Psychic,"['Quark Drive', 'Quark Drive', 'Quark Drive']",9.0,NaN,72,100,90,122,108,98,1.0,156.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,590,10.0,NaN
1058,1024,Terapagos,No hay,Normal,No more types,"['Tera Shift', 'Tera Shift', 'Tera Shift']",9.0,NaN,65,85,90,65,85,60,0.0,6.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,450,255.0,NaN
1059,1024,Terapagos,No hay,Normal,No more types,"['Tera Shift', 'Tera Shift', 'Tera Shift']",9.0,NaN,65,85,90,65,85,60,0.0,6.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,450,255.0,NaN
1060,1024,Terapagos,No hay,Normal,No more types,"['Tera Shift', 'Tera Shift', 'Tera Shift']",9.0,NaN,65,85,90,65,85,60,0.0,6.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,450,255.0,NaN


In [127]:
# Juntamos los types en una columna

pokemon_casi["Combination"] = pokemon_casi["Base_type"] + ', ' + pokemon_casi["Secondary_type"]
len(set(pokemon_casi["Combination"]))

213

In [128]:
# No NAN DF

pokemon_no_na = pokemon_casi[~pokemon_casi["Against_bug"].isna()]


In [87]:
# Cogemos la media por against
all_types = set(pokemon_no_na["Combination"])
# Creamos un diccionario vacio para ir añadiendo las medias de against
defense_mapping = {}


for comb in all_types:
    # Split for types
    base, secondary = comb.split(', ')
    # Mean for each against type in the DF
    against_bug_mean = pokemon_no_na[(pokemon_no_na["Base_type"] == base) & (pokemon_no_na["Secondary_type"] == secondary)]["Against_bug"].mean()
    against_dark_mean = pokemon_no_na[(pokemon_no_na["Base_type"] == base) & (pokemon_no_na["Secondary_type"] == secondary)]["Against_dark"].mean()
    against_fire_mean = pokemon_no_na[(pokemon_no_na["Base_type"] == base) & (pokemon_no_na["Secondary_type"] == secondary)]["Against_fire"].mean()
    against_electric_mean = pokemon_no_na[(pokemon_no_na["Base_type"] == base) & (pokemon_no_na["Secondary_type"] == secondary)]["Against_electric"].mean()
    against_dragon_mean = pokemon_no_na[(pokemon_no_na["Base_type"] == base) & (pokemon_no_na["Secondary_type"] == secondary)]["Against_dragon"].mean()
    against_fairy_mean = pokemon_no_na[(pokemon_no_na["Base_type"] == base) & (pokemon_no_na["Secondary_type"] == secondary)]["Against_fairy"].mean()
    against_fight_mean = pokemon_no_na[(pokemon_no_na["Base_type"] == base) & (pokemon_no_na["Secondary_type"] == secondary)]["Against_fight"].mean()
    against_fly_mean = pokemon_no_na[(pokemon_no_na["Base_type"] == base) & (pokemon_no_na["Secondary_type"] == secondary)]["Against_flying"].mean()
    against_ghost_mean = pokemon_no_na[(pokemon_no_na["Base_type"] == base) & (pokemon_no_na["Secondary_type"] == secondary)]["Against_ghost"].mean()
    against_grass_mean = pokemon_no_na[(pokemon_no_na["Base_type"] == base) & (pokemon_no_na["Secondary_type"] == secondary)]["Against_grass"].mean()
    against_ground_mean = pokemon_no_na[(pokemon_no_na["Base_type"] == base) & (pokemon_no_na["Secondary_type"] == secondary)]["Against_ground"].mean()
    against_ice_mean = pokemon_no_na[(pokemon_no_na["Base_type"] == base) & (pokemon_no_na["Secondary_type"] == secondary)]["Against_ice"].mean()
    against_normal_mean = pokemon_no_na[(pokemon_no_na["Base_type"] == base) & (pokemon_no_na["Secondary_type"] == secondary)]["Against_normal"].mean()
    against_poison_mean = pokemon_no_na[(pokemon_no_na["Base_type"] == base) & (pokemon_no_na["Secondary_type"] == secondary)]["Against_poison"].mean()
    against_psy_mean = pokemon_no_na[(pokemon_no_na["Base_type"] == base) & (pokemon_no_na["Secondary_type"] == secondary)]["Against_psychic"].mean()
    against_rock_mean = pokemon_no_na[(pokemon_no_na["Base_type"] == base) & (pokemon_no_na["Secondary_type"] == secondary)]["Against_rock"].mean()
    against_steel_mean = pokemon_no_na[(pokemon_no_na["Base_type"] == base) & (pokemon_no_na["Secondary_type"] == secondary)]["Against_steel"].mean()
    against_water_mean = pokemon_no_na[(pokemon_no_na["Base_type"] == base) & (pokemon_no_na["Secondary_type"] == secondary)]["Against_water"].mean()

    # Append it to the dictionary
    defense_mapping[comb]["bug"] = against_bug_mean
    defense_mapping[comb]["dark"] = against_dark_mean
    defense_mapping[comb]["fire"] = against_fire_mean
    defense_mapping[comb]["electric"] = against_electric_mean
    defense_mapping[comb]["dragon"] = against_dragon_mean
    defense_mapping[comb]["fairy"] = against_fairy_mean
    defense_mapping[comb]["fight"] = against_fight_mean
    defense_mapping[comb]["flying"] = against_fly_mean
    defense_mapping[comb]["ghost"] = against_ghost_mean
    defense_mapping[comb]["grass"] = against_grass_mean
    defense_mapping[comb]["ground"] = against_ground_mean
    defense_mapping[comb]["ice"] = against_ice_mean
    defense_mapping[comb]["normal"] = against_normal_mean
    defense_mapping[comb]["poison"] = against_poison_mean
    defense_mapping[comb]["psychic"] = against_psy_mean
    defense_mapping[comb]["rock"] = against_rock_mean
    defense_mapping[comb]["steel"] = against_steel_mean
    defense_mapping[comb]["water"] = against_water_mean
    
print(defense_mapping)



KeyError: 'Ground, Fire'

In [133]:
against_bug_mean

1.0

In [ ]:
pokemon_casi[(pokemon_casi["Base_type"] == 'Ground') & (pokemon_casi["Secondary_type"] == 'Fire')]

,Pokedex_number,Name,Evolution,Base_type,Secondary_type,Abilities,Generation,Legendary,Attack,Defense,Hp,Sp_attack,Sp_defense,Speed,Height_m,Weight_kg,Against_bug,Against_dark,Against_dragon,Against_electric,Against_fairy,Against_fight,Against_fire,Against_flying,Against_ghost,Against_grass,Against_ground,Against_ice,Against_normal,Against_poison,Against_psychic,Against_rock,Against_steel,Against_water,Base_egg_steps,Base_happiness,Base_total,Capture_rate,Experience_growth,Combination
104,105,Marowak,Unknown,Ground,Fire,"['Rock Head', 'Lightningrod', 'Battle Armor', ...",1.0,0.0,80,110,60,50,80,45,NaN,NaN,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,2.0,1.0,0.5,1.0,0.5,1.0,2.0,5120.0,70.0,425,75,1000000.0,"Ground, Fire"


In [ ]:
# Creamos una nueva columna con el numero de habilidades que tiene cada pokemon para pasarlo al modelo
def count_abilities(x):
    return len(x.split(","))

pokemon_casi['Number_of_Abilities'] = pokemon_casi['Abilities'].apply(count_abilities)

pokemon_casi['Number_of_Abilities']

In [ ]:
pokemon_casi.tail()

,Pokedex_number,Name,Evolution,Base_type,Secondary_type,Abilities,Generation,Legendary,Attack,Defense,Hp,Sp_attack,Sp_defense,Speed,Height_m,Weight_kg,Against_bug,Against_dark,Against_dragon,Against_electric,Against_fairy,Against_fight,Against_fire,Against_flying,Against_ghost,Against_grass,Against_ground,Against_ice,Against_normal,Against_poison,Against_psychic,Against_rock,Against_steel,Against_water,Base_egg_steps,Base_happiness,Base_total,Capture_rate,Experience_growth
1039,1021,Raging Bolt,Final Form,Electric,Dragon,NaN,9.0,0.0,73,91,125,137,89,75,1.196205,64.576563,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,590,NaN,NaN
1040,1022,Iron Boulder,Final Form,Rock,Psychic,NaN,9.0,0.0,120,80,90,68,108,124,1.196205,64.576563,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,590,NaN,NaN
1041,1023,Iron Crown,Final Form,Steel,Psychic,NaN,9.0,0.0,72,100,90,122,108,98,1.196205,64.576563,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,590,NaN,NaN
1042,1024,Terapagos,Final Form,Normal,No more types,NaN,9.0,1.0,65,85,90,65,85,60,1.196205,64.576563,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,450,NaN,NaN
1043,1025,Pecharunt,Final Form,Poison,Ghost,NaN,9.0,1.0,88,160,88,88,88,88,1.196205,64.576563,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,600,NaN,NaN


In [ ]:
# OBJETIVOS

# ULTIMATE POKEDEX CON STREAMLIT
# Primera pestaña introduccion

# MAPEO DE DONDE SE ENCUENTRAN LOS POKEMONS (GEOJSON MAPA POKEMON?/MAPA REAL?) PARA VER DONDE CAPTURARLOS
# MONTAR TU PROPIO EQUIPO Y PODER HACER PREDICTOR DE BATALLAS
# CLASIFICADOR DE POKEMON SEGUN FEATURES? Y RECOMENDAR, TAMBIEN LINKEARLO CON CHATGTP Y PODER CREAR UNA IMAGEN(TIPO, LEGENDARIO, GENERACION, ESTADISTICAS ETC..)


# CONSTRUIR UN POKEMON A PARTIR DE IMAGENES USANDO ALGORITMO DE CLASIFICACION? (DEMASIADO LOCO CREO) - HACER PESTAÑA CON QUIZZ
# ANALISIS DESCRIPTIVO DE LOS DATOS

# AÑADIR LLM PARA QUE STREAMLIT SEA INTERACTIVO? SI SOBRA TIEMPO (NO ES UNA PRIORIDAD)
# CREAR UN POKEMON VIA CHATGPT